In [17]:
import pandas as pd
import numpy as np
import re
import ast
import joblib
import os

from sklearn.feature_extraction.text import TfidfVectorizer

In [51]:
job_df = pd.read_csv("dataset/cleaned_dataset_job.csv")

job_df.head()

,company,job_title,job_type,experience_level,skills
0,SGS,clinical data analyst,Full Time,Entry-level,"['genetics', 'sas', 'computer science', 'data ..."
1,Ocorian,aml/cft & data analyst,Full Time,Entry-level,"['agile', 'security', 'data management', 'fina..."
2,Cricut,machine learning engineer,Full Time,No Experience,"['agile', 'deep learning', 'architecture', 'aw..."
3,Bosch Group,application developer & data analyst,Full Time,Entry-level,"['power bi', 'oracle', 'rd', 'industrial', 'en..."
4,Publicis Groupe,data engineer full time (public sector) usa,Full Time,Mid-level,"['data pipelines', 'azure', 'aws', 'computer s..."


In [19]:
job_df = job_df.drop_duplicates().copy()
job_df.shape

(2900, 5)

In [20]:
# Parse skills
def parse_skills(skill_text):
    try:
        return ast.literal_eval(skill_text)
    except:
        return []

job_df["skills_list"] = job_df["skills"].apply(parse_skills)

In [21]:
# Combine text features dari Job_title + Experience_level + Skills_list
job_df["combined_text"] = (
    job_df["job_title"].astype(str) + " " +
    job_df["experience_level"].astype(str) + " " +
    job_df["skills_list"].astype(str)
    )

In [22]:
# Text Cleaning
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [23]:
job_df["combined_cleaned"] = ( job_df["combined_text"].apply(clean_text))

In [24]:
# TF-IDF
tfidf_job = TfidfVectorizer(max_features=3000, ngram_range=(1,2))

In [25]:
X_job = tfidf_job.fit_transform(
    job_df["combined_cleaned"]
)

In [27]:
X_job.shape

(2900, 3000)

In [43]:
os.makedirs("models", exist_ok=True)
joblib.dump(tfidf_job, "models/tfidf_job_dataset.pkl")

['models/tfidf_job_dataset.pkl']

In [49]:
import os

os.makedirs("dataset", exist_ok=True)

job_df.to_csv(
    "dataset/job_dataset_processed.csv",
    index=False
)

## Preprocessing Job Dataset

Dataset job posting berhasil diproses dan dibersihkan untuk mendukung fitur skill gap analysis pada sistem AI Career Navigator. Informasi posisi pekerjaan, level pengalaman, dan daftar skill berhasil digabungkan menjadi representasi teks terstruktur dan diubah menjadi fitur numerik menggunakan TF-IDF.